ЧАСТЬ I. АРХИТЕКТУРА
1.1. Общая схема системы
```text
┌─────────────────────────────────────────────────────────────────┐

│                        ПОЛЬЗОВАТЕЛЬ                             │

│                     (Браузер)                                   │

└──────────────────────────┬──────────────────────────────────────┘

                           │ HTTP/REST/JSON

                           ▼

┌─────────────────────────────────────────────────────────────────┐

│                   FRONTEND (React + TypeScript)                 │

│                                                                 │

│  ┌──────────┐ ┌──────────────┐ ┌────────────┐ ┌──────────────┐  │

│  │Dashboard │ │ Prediction   │ │Vulnerability│ │   Threat    │  │

│  │  Page    │ │    Page      │ │    Page     │ │  Catalog    │  │

│  └──────────┘ └──────────────┘ └────────────┘ └──────────────┘  │

│                                                                 │

│  ┌────────────────────────────────────────────────────────────┐ │

│  │              API Client (RTK Query / Axios)                │ │

│  └────────────────────────────────────────────────────────────┘ │

└──────────────────────────┬──────────────────────────────────────┘

                           │ REST JSON

                           ▼

┌─────────────────────────────────────────────────────────────────┐

│                 BACKEND (FastAPI, порт 8000)                    │

│                                                                 │

│  ┌───────────────────────────────────────────────────────────┐  │

│  │                    API LAYER (Routers)                    │  │

│  │  /health  /analytics  /predict  /threats  /recommendations│  │

│  └───────────────────────┬───────────────────────────────────┘  │

│                          │                                      │

│  ┌───────────────────────▼───────────────────────────────────┐  │

│  │                  SERVICE LAYER                            │  │

│  │  AnalyticsService  PredictionService  RecommendationService│ │

│  └──────┬──────────────────┬─────────────────────┬───────────┘  │

│         │                  │                     │              │

│         ▼                  ▼                     ▼              │

│  ┌────────────┐   ┌───────────────┐   ┌──────────────────┐      │

│  │ DB Layer   │   │  ML Module    │   │ Recommendation   │      │

│  │ SQLAlchemy │   │  (встроенный) │   │    Engine        │      │

│  │ Repository │   │               │   │                  │      │

│  └─────┬──────┘   │ ┌───────────┐ │   └──────────────────┘      │

│        │          │ │ CatBoost  │ │                             │

│        │          │ │ LightGBM  │ │                             │

│        │          │ │ Scoring   │ │                             │

│        │          │ │ SHAP      │ │                             │

│        │          │ └───────────┘ │                             │

│        │          └───────────────┘                             │

└────────┼────────────────────────────────────────────────────────┘

         │ SQL

         ▼

┌─────────────────────────────────────────────────────────────────┐

│              POSTGRESQL (порт 5432)                             │

│                                                                 │

│  ┌──────────┐ ┌──────────┐ ┌───────────────────┐                │

│  │ threats  │ │incidents │ │enterprise_profiles│                │

│  │ (227)    │ │ (2000)   │ │    (canonical)    │                │

│  └──────────┘ └──────────┘ └───────────────────┘                │

│  ┌──────────────┐ ┌───────────────┐ ┌──────────────┐            │

│  │ predictions  │ │recommendations│ │demo_scenarios│            │

│  │   (log)      │ │  (справочник) │ │  (presets)   │            │

│  └──────────────┘ └───────────────┘ └──────────────┘            │

└─────────────────────────────────────────────────────────────────┘
```

этап 1: Фиксируем JSON-контракты API

        ├── Frontend работает на МОКАХ (заглушках)

        ├── Backend пишет реальные эндпоинты

        ├── ML обучает модели в Jupyter

        └── DevOps поднимает Docker + БД

этап 2: Backend заменяет моки на реальные вызовы ML

         Frontend переключается на реальный API

         Ничего не ломается — контракты те же


# ER-ДИАГРАММА БАЗЫ ДАННЫХ

2.1. Визуальная схема
┌─────────────────────────┐       ┌──────────────────────────────┐

│       threats            │       │        incidents              │

│─────────────────────────│       │──────────────────────────────│

│ PK id          SERIAL   │       │ PK id             SERIAL     │

│    ubi_code    VARCHAR(10)│◄─┐   │ FK threat_code    INT        │──┐

│    name        TEXT       │  │   │ FK enterprise_code VARCHAR(10)│  │

│    description TEXT       │  │   │    company_type   VARCHAR(50) │  │

│    source_type VARCHAR(50)│  │   │    host_count     INT         │  │

│    impact_object TEXT     │  │   │    success        BOOLEAN     │  │

│    conf_violation  INT    │  │   │    region         TEXT         │  │

│    integ_violation INT    │  └───│    incident_date  DATE         │  │

│    avail_violation INT    │      │    regional_time  TIME         │  │

│    include_date DATE      │      │    incident_hour  INT          │  │

│    updated_at  TIMESTAMP  │      │    day_of_week    INT          │  │

│    status      VARCHAR(20)│      │    month          INT          │  │

│    comment     TEXT       │      │    season         VARCHAR(10)  │  │

└─────────────────────────┘       │    is_canonical   BOOLEAN      │  │

                                   │    created_at     TIMESTAMP    │  │

                                   └──────────────────────────────┘  │

                                                                      │

┌──────────────────────────────┐                                      │

│    enterprise_profiles        │                                      │

│──────────────────────────────│                                      │

│ PK id               SERIAL   │                                      │

│ UQ enterprise_code  VARCHAR(10)│◄────────────────────────────────────┘

│    type_canonical   VARCHAR(50)│

│    region_canonical TEXT       │       ┌──────────────────────────────┐

│    host_count_canonical INT   │       │       predictions             │

│    risk_score       FLOAT     │       │──────────────────────────────│

│    last_incident_date DATE    │       │ PK id              SERIAL    │

│    incident_count   INT       │       │    request_id      UUID      │

│    success_count    INT       │◄──────│ FK enterprise_code VARCHAR(10)│

│    created_at       TIMESTAMP │       │    prediction_ts   TIMESTAMP │

│    updated_at       TIMESTAMP │       │    incident_prob   FLOAT     │

└──────────────────────────────┘       │    attack_time_bucket VARCHAR│

                                        │    attack_hour      INT      │

┌──────────────────────────────┐       │ FK predicted_threat INT       │

│     recommendations           │       │    predicted_object VARCHAR  │

│──────────────────────────────│       │    vuln_level      VARCHAR   │

│ PK id              SERIAL    │       │    vuln_score      FLOAT     │

│ UQ rec_code        VARCHAR(20)│◄──────│ FK rec_code        VARCHAR(20)│

│ FK threat_code     INT        │──┐    │    model_version   VARCHAR   │

│    title           TEXT       │  │    │    features_json   JSONB     │

│    description     TEXT       │  │    │    explanation_json JSONB    │

│    priority        INT        │  │    └──────────────────────────────┘

│    target_object   TEXT       │  │

│    vuln_level      VARCHAR(20)│  │    ┌──────────────────────────────┐

│    source_standard VARCHAR(50)│  │    │      demo_scenarios           │

│    is_active       BOOLEAN    │  │    │──────────────────────────────│

└──────────────────────────────┘  │    │ PK id              SERIAL    │

         │                         │    │    scenario_name   VARCHAR   │

         │                         │    │    description     TEXT      │

         └─────────────────────────┘    │    request_payload JSONB    │

                                        │    expected_result JSONB    │

                                        └──────────────────────────────┘
2.2. Связи между таблицами
threats.ubi_code ──────────< incidents.threat_code

                              (один ко многим: одна угроза — много инцидентов)

enterprise_profiles.enterprise_code ──────────< incidents.enterprise_code

                              (одно предприятие — много инцидентов)

enterprise_profiles.enterprise_code ──────────< predictions.enterprise_code

                              (одно предприятие — много прогнозов)

threats.ubi_code ──────────< recommendations.threat_code

                              (одна угроза — несколько рекомендаций)

threats.ubi_code ──────────< predictions.predicted_threat

                              (прогноз ссылается на угрозу)

recommendations.rec_code ──────────< predictions.rec_code

                              (прогноз ссылается на рекомендацию)
2.3. SQL-скрипт создания таблиц
-- db/init/001_create_tables.sql

-- 1. Справочник угроз ФСТЭК (из thrlist.xlsx)

CREATE TABLE threats (

    id            SERIAL PRIMARY KEY,

    ubi_code      VARCHAR(10) UNIQUE NOT NULL,

    name          TEXT NOT NULL,

    description   TEXT,

    source_type   VARCHAR(100),

    impact_object TEXT,

    conf_violation    INT DEFAULT 0,

    integ_violation   INT DEFAULT 0,

    avail_violation   INT DEFAULT 0,

    include_date  DATE,

    updated_at    TIMESTAMP,

    status        VARCHAR(30) DEFAULT 'Опубликована',

    comment       TEXT

);

-- 2. Канонические профили предприятий

CREATE TABLE enterprise_profiles (

    id                  SERIAL PRIMARY KEY,

    enterprise_code     VARCHAR(10) UNIQUE NOT NULL,

    type_canonical      VARCHAR(50) NOT NULL,

    region_canonical    TEXT NOT NULL,

    host_count_canonical INT NOT NULL,

    risk_score          FLOAT DEFAULT 0.0,

    last_incident_date  DATE,

    incident_count      INT DEFAULT 0,

    success_count       INT DEFAULT 0,

    created_at          TIMESTAMP DEFAULT NOW(),

    updated_at          TIMESTAMP DEFAULT NOW()

);

-- 3. Инциденты (из incidents_2000.csv, очищенные)

CREATE TABLE incidents (

    id              SERIAL PRIMARY KEY,

    enterprise_code VARCHAR(10) NOT NULL

        REFERENCES enterprise_profiles(enterprise_code),

    company_type    VARCHAR(50),

    host_count      INT,

    threat_code     VARCHAR(10)

        REFERENCES threats(ubi_code),

    success         BOOLEAN NOT NULL,

    region          TEXT,

    incident_date   DATE NOT NULL,

    regional_time   TIME,

    incident_hour   INT,

    day_of_week     INT,

    month           INT,

    season          VARCHAR(10),

    is_canonical    BOOLEAN DEFAULT TRUE,

    created_at      TIMESTAMP DEFAULT NOW()

);

-- 4. Справочник рекомендаций

CREATE TABLE recommendations (

    id              SERIAL PRIMARY KEY,

    rec_code        VARCHAR(20) UNIQUE NOT NULL,

    threat_code     VARCHAR(10)

        REFERENCES threats(ubi_code),

    title           TEXT NOT NULL,

    description     TEXT,

    priority        INT DEFAULT 5,

    target_object   TEXT,

    vuln_level      VARCHAR(20),

    source_standard VARCHAR(50),

    is_active       BOOLEAN DEFAULT TRUE

);

-- 5. Лог прогнозов

CREATE TABLE predictions (

    id                SERIAL PRIMARY KEY,

    request_id        UUID NOT NULL DEFAULT gen_random_uuid(),

    enterprise_code   VARCHAR(10)

        REFERENCES enterprise_profiles(enterprise_code),

    prediction_ts     TIMESTAMP DEFAULT NOW(),

    incident_prob     FLOAT,

    attack_time_bucket VARCHAR(20),

    attack_hour       INT,

    predicted_threat  VARCHAR(10)

        REFERENCES threats(ubi_code),

    predicted_object  VARCHAR(50),

    vuln_level        VARCHAR(20),

    vuln_score        FLOAT,

    rec_code          VARCHAR(20)

        REFERENCES recommendations(rec_code),

    model_version     VARCHAR(10) DEFAULT 'v1',

    features_json     JSONB,

    explanation_json  JSONB

);

-- 6. Демо-сценарии

CREATE TABLE demo_scenarios (

    id              SERIAL PRIMARY KEY,

    scenario_name   VARCHAR(100) NOT NULL,

    description     TEXT,

    request_payload JSONB NOT NULL,

    expected_result JSONB

);

-- db/init/002_indexes.sql

CREATE INDEX idx_incidents_threat      ON incidents(threat_code);

CREATE INDEX idx_incidents_enterprise  ON incidents(enterprise_code);

CREATE INDEX idx_incidents_region      ON incidents(region);

CREATE INDEX idx_incidents_date        ON incidents(incident_date);

CREATE INDEX idx_incidents_success     ON incidents(success);

CREATE INDEX idx_predictions_ts        ON predictions(prediction_ts);

CREATE INDEX idx_predictions_enterprise ON predictions(enterprise_code);

CREATE INDEX idx_threats_ubi           ON threats(ubi_code);

CREATE INDEX idx_recommendations_threat ON recommendations(threat_code);

# ПОЛНЫЙ СПИСОК API ENDPOINTS

3.1. Сводная таблица
#
Метод
Endpoint
Назначение
SLA
1
GET
/api/v1/health
Проверка работоспособности
<10ms
2
GET
/api/v1/analytics/summary
Общая статистика для дашборда
<100ms
3
GET
/api/v1/analytics/timeseries
Инциденты по периодам
<100ms
4
GET
/api/v1/analytics/regions
Статистика по регионам
<100ms
5
GET
/api/v1/analytics/enterprise-types
Статистика по отраслям
<100ms
6
GET
/api/v1/threats
Каталог угроз ФСТЭК
<100ms
7
GET
/api/v1/threats/{ubi_code}
Детали одной угрозы
<50ms
8
POST
/api/v1/predict
Главный прогноз (6 моделей)
<300ms
9
GET
/api/v1/recommendations
Список рекомендаций
<100ms
10
GET
/api/v1/scenarios/demo
Список демо-сценариев
<50ms
11
POST
/api/v1/scenarios/demo/{id}/run
Запуск демо-сценария
<300ms



3.2. Детальные схемы каждого endpoint

ENDPOINT 1: Health Check
GET /api/v1/health

Описание: Проверка что backend, БД и ML-модели загружены.

Request: нет параметров

Response 200:

{

    "status": "ok",

    "database": "connected",

    "models_loaded": 6,

    "model_version": "v1",

    "timestamp": "2025-05-20T14:30:00Z"

}

Response 503:

{

    "status": "degraded",

    "database": "connected",

    "models_loaded": 4,

    "errors": ["model_vulnerability not loaded"]

}


ENDPOINT 2: Analytics Summary
GET /api/v1/analytics/summary

Описание: Агрегированная статистика для KPI-карточек на дашборде.

Query Parameters:

Параметр
Тип
Обязательный
Описание
region
string
нет
Фильтр по региону
enterprise_type
string
нет
Фильтр по отрасли
date_from
string (YYYY-MM-DD)
нет
Начало периода
date_to
string (YYYY-MM-DD)
нет
Конец периода


Request пример:

GET /api/v1/analytics/summary?region=Москва&date_from=2024-01-01

Response 200:

{

    "totals": {

        "total_incidents": 2000,

        "successful_incidents": 943,

        "success_rate": 0.4715,

        "unique_threats": 187,

        "unique_regions": 89,

        "unique_enterprises": 412,

        "avg_host_count": 892

    },

    "top_threats": [

        {

            "threat_code": "190",

            "threat_name": "Угроза внедрения вредоносного кода в BIOS",

            "count": 83,

            "success_rate": 0.61

        },

        {

            "threat_code": "152",

            "threat_name": "Угроза перехвата управления загрузкой",

            "count": 77,

            "success_rate": 0.54

        },

        {

            "threat_code": "163",

            "threat_name": "Угроза подмены программного обеспечения",

            "count": 65,

            "success_rate": 0.48

        }

    ],

    "top_regions": [

        {

            "region": "Хабаровский край",

            "count": 29,

            "success_rate": 0.55

        },

        {

            "region": "Москва",

            "count": 27,

            "success_rate": 0.41

        }

    ],

    "top_enterprise_types": [

        {

            "enterprise_type": "Медицина",

            "count": 121,

            "success_rate": 0.67

        },

        {

            "enterprise_type": "НКО",

            "count": 98,

            "success_rate": 0.52

        }

    ],

    "filters_applied": {

        "region": "Москва",

        "date_from": "2024-01-01",

        "date_to": null,

        "enterprise_type": null

    }

}


ENDPOINT 3: Analytics Timeseries
GET /api/v1/analytics/timeseries

Описание: Данные для графика инцидентов по времени.

Query Parameters:

Параметр
Тип
Обязательный
Описание
granularity
string
нет
month (default), week, quarter
region
string
нет
Фильтр по региону
enterprise_type
string
нет
Фильтр по отрасли


Response 200:

{

    "granularity": "month",

    "series": [

        {

            "period": "2023-01",

            "total": 41,

            "successful": 19,

            "failed": 22

        },

        {

            "period": "2023-02",

            "total": 37,

            "successful": 21,

            "failed": 16

        },

        {

            "period": "2023-03",

            "total": 52,

            "successful": 28,

            "failed": 24

        }

    ]

}


ENDPOINT 4: Analytics by Regions
GET /api/v1/analytics/regions

Описание: Статистика инцидентов в разрезе регионов (для карты или bar chart).

Query Parameters:

Параметр
Тип
Обязательный
Описание
top_n
int
нет
Количество регионов (default 20)
sort_by
string
нет
count (default), success_rate


Response 200:

{

    "regions": [

        {

            "region": "Хабаровский край",

            "total_incidents": 29,

            "successful": 16,

            "success_rate": 0.55,

            "unique_enterprises": 8,

            "avg_host_count": 1120,

            "top_threat_code": "190"

        },

        {

            "region": "Москва",

            "total_incidents": 27,

            "successful": 11,

            "success_rate": 0.41,

            "unique_enterprises": 15,

            "avg_host_count": 2340,

            "top_threat_code": "152"

        }

    ],

    "total_regions": 89

}


ENDPOINT 5: Analytics by Enterprise Types
GET /api/v1/analytics/enterprise-types

Описание: Статистика по отраслям.

Response 200:

{

    "enterprise_types": [

        {

            "enterprise_type": "Медицина",

            "total_incidents": 121,

            "successful": 81,

            "success_rate": 0.67,

            "avg_host_count": 1450,

            "unique_regions": 34,

            "risk_label": "critical"

        },

        {

            "enterprise_type": "НКО",

            "total_incidents": 98,

            "successful": 51,

            "success_rate": 0.52,

            "avg_host_count": 320,

            "unique_regions": 28,

            "risk_label": "high"

        }

    ]

}


ENDPOINT 6: Threat Catalog (список)
GET /api/v1/threats

Описание: Каталог угроз ФСТЭК с поиском и пагинацией.

Query Parameters:

Параметр
Тип
Обязательный
Описание
search
string
нет
Поиск по названию/описанию
impact_object
string
нет
Фильтр по объекту воздействия
page
int
нет
Номер страницы (default 1)
per_page
int
нет
Записей на странице (default 20)


Response 200:

{

    "items": [

        {

            "ubi_code": "001",

            "name": "Угроза автоматического распространения вредоносного кода в грид-системе",

            "impact_object": "Ресурсные центры грид-системы",

            "conf_violation": 1,

            "integ_violation": 1,

            "avail_violation": 1,

            "status": "Опубликована",

            "incident_count": 12

        },

        {

            "ubi_code": "002",

            "name": "Угроза агрегирования данных, передаваемых датчиками",

            "impact_object": "Датчики, контроллеры",

            "conf_violation": 1,

            "integ_violation": 0,

            "avail_violation": 0,

            "status": "Опубликована",

            "incident_count": 5

        }

    ],

    "total": 227,

    "page": 1,

    "per_page": 20,

    "pages": 12

}


ENDPOINT 7: Threat Detail
GET /api/v1/threats/{ubi_code}

Описание: Полная карточка одной угрозы с историей инцидентов.

Path Parameters:

Параметр
Тип
Описание
ubi_code
string
Код угрозы ФСТЭК (например "190")


Response 200:

{

    "ubi_code": "190",

    "name": "Угроза внедрения вредоносного кода в BIOS",

    "description": "Угроза заключается в возможности внедрения в BIOS...",

    "source_type": "Внешний нарушитель с высоким потенциалом",

    "impact_object": "Микропрограммное обеспечение BIOS/UEFI",

    "conf_violation": 1,

    "integ_violation": 1,

    "avail_violation": 1,

    "status": "Опубликована",

    "include_date": "2015-02-01",

    "updated_at": "2023-06-15T00:00:00Z",

    "incident_stats": {

        "total_incidents": 83,

        "successful": 51,

        "success_rate": 0.61,

        "most_affected_region": "Хабаровский край",

        "most_affected_type": "Медицина"

    },

    "related_recommendations": [

        {

            "rec_code": "REC-BIOS-001",

            "title": "Обновление прошивок BIOS/UEFI",

            "priority": 1

        },

        {

            "rec_code": "REC-BIOS-002",

            "title": "Включение Secure Boot",

            "priority": 2

        }

    ]

}

Response 404:

{

    "error": {

        "code": "THREAT_NOT_FOUND",

        "message": "Угроза с кодом 999 не найдена"

    }

}


ENDPOINT 8: Predict (ГЛАВНЫЙ)
POST /api/v1/predict

Описание: Запуск всех 6 моделей прогнозирования. Это центральный endpoint системы.

Request Body:

{

    "enterprise_type": "Медицина",

    "host_count": 1500,

    "region": "Республика Саха (Якутия)",

    "observation_date": "2025-05-20",

    "additional_context": {

        "internet_exposed": true,

        "has_critical_data": true,

        "segmentation_level": "low"

    }

}

Валидация полей (Pydantic):

Поле
Тип
Обязательное
Ограничения
enterprise_type
string
да
Enum: Медицина, НКО, Образование, Промышленность, Госуправление, Финансы, Транспорт, Энергетика, Телеком, Другое
host_count
int
да
min=1, max=100000
region
string
да
Из справочника регионов
observation_date
string
да
Формат YYYY-MM-DD
additional_context
object
нет
Дополнительные признаки


Response 200:

{

    "request_id": "b8e8b4ce-4d78-4f2f-89f4-1af1f551d311",

    "model_version": "v1",

    "inference_time_ms": 127,

    "incident_prediction": {

        "will_happen": true,

        "probability": 0.85,

        "confidence_level": "high",

        "confidence_label": "Высокая уверенность"

    },

    "attack_time_prediction": {

        "time_bucket": "night",

        "time_bucket_label": "Ночь (00:00 — 06:00)",

        "probable_hour": 2,

        "probable_day_of_week": "friday",

        "probable_day_label": "Пятница",

        "probable_season": "summer"

    },

    "threat_prediction": {

        "primary": {

            "threat_code": "190",

            "threat_name": "Угроза внедрения вредоносного кода в BIOS",

            "probability": 0.52

        },

        "top_3": [

            {

                "threat_code": "190",

                "threat_name": "Угроза внедрения вредоносного кода в BIOS",

                "probability": 0.52

            },

            {

                "threat_code": "152",

                "threat_name": "Угроза перехвата управления загрузкой",

                "probability": 0.19

            },

            {

                "threat_code": "163",

                "threat_name": "Угроза подмены программного обеспечения",

                "probability": 0.11

            }

        ]

    },

    "target_object_prediction": {

        "primary": {

            "object_code": "bios_firmware",

            "object_name": "Микропрограммное обеспечение BIOS/UEFI",

            "probability": 0.62

        },

        "top_3": [

            {

                "object_code": "bios_firmware",

                "object_name": "Микропрограммное обеспечение BIOS/UEFI",

                "probability": 0.62

            },

            {

                "object_code": "server",

                "object_name": "Серверное оборудование",

                "probability": 0.21

            },

            {

                "object_code": "network",

                "object_name": "Сетевой сегмент",

                "probability": 0.09

            }

        ]

    },

    "vulnerability_assessment": {

        "level": "critical",

        "level_label": "Критический",

        "score": 0.87,

        "factors": [

            "Высокая вероятность успешной атаки (85%)",

            "Большое количество хостов (1500)",

            "Регион с повышенной активностью атак",

            "Отрасль Медицина — в зоне повышенного риска"

        ]

    },

    "recommendations": [

        {

            "rec_code": "REC-BIOS-001",

            "title": "Обновить прошивки BIOS/UEFI на всех хостах",

            "description": "Проверить наличие обновлений BIOS у производителей оборудования. Применить патчи на критичных серверах в первую очередь.",

            "priority": 1,

            "priority_label": "Критический",

            "related_threat": "190"

        },

        {

            "rec_code": "REC-NET-003",

            "title": "Усилить сегментацию сети",

            "description": "Изолировать критичные сегменты с медицинскими данными. Настроить межсетевые экраны между VLAN.",

            "priority": 2,

            "priority_label": "Высокий",

            "related_threat": "190"

        },

        {

            "rec_code": "REC-MON-001",

            "title": "Усилить ночной мониторинг (00:00-06:00)",

            "description": "Модель прогнозирует атаку в ночное время. Рекомендуется усилить дежурную смену SOC в указанный период.",

            "priority": 3,

            "priority_label": "Средний",

            "related_threat": null

        }

    ],

    "explanations": {

        "method": "SHAP",

        "top_features": [

            {

                "feature": "enterprise_type",

                "display_name": "Отрасль: Медицина",

                "impact": 0.21,

                "direction": "increases_risk",

                "explanation": "Медицинские учреждения атакуются на 67% чаще среднего"

            },

            {

                "feature": "host_count",

                "display_name": "Количество хостов: 1500",

                "impact": 0.18,

                "direction": "increases_risk",

                "explanation": "Большая поверхность атаки увеличивает вероятность"

            },

            {

                "feature": "region",

                "display_name": "Регион: Якутия",

                "impact": 0.12,

                "direction": "increases_risk",

                "explanation": "В данном регионе зафиксирован рост инцидентов"

            },

            {

                "feature": "season",

                "display_name": "Сезон: Лето",

                "impact": 0.08,

                "direction": "increases_risk",

                "explanation": "Летний период характеризуется снижением бдительности персонала"

            },

            {

                "feature": "segmentation_level",

                "display_name": "Сегментация: Низкая",

                "impact": 0.07,

                "direction": "increases_risk",

                "explanation": "Отсутствие сегментации облегчает горизонтальное перемещение"

            }

        ]

    },

    "business_impact": {

        "estimated_damage_rub": 15200000,

        "damage_label": "~15.2 млн руб.",

        "calculation_basis": "Средний ущерб от успешной атаки на медицинское учреждение с 1500+ хостами по данным НКЦКИ"

    }

}

Response 422 (ошибка валидации):

{

    "error": {

        "code": "VALIDATION_ERROR",

        "message": "Ошибка валидации входных данных",

        "details": [

            {

                "field": "host_count",

                "message": "Значение должно быть больше 0",

                "received": -5

            },

            {

                "field": "region",

                "message": "Регион 'Луна' не найден в справочнике",

                "received": "Луна"

            }

        ]

    }

}

Response 500 (ошибка ML):

{

    "error": {

        "code": "ML_INFERENCE_ERROR",

        "message": "Аналитический модуль временно недоступен",

        "fallback_available": true

    }

}


ENDPOINT 9: Recommendations List
GET /api/v1/recommendations

Описание: Справочник всех рекомендаций с фильтрацией.

Query Parameters:

Параметр
Тип
Обязательный
Описание
threat_code
string
нет
Фильтр по коду угрозы
vuln_level
string
нет
low, medium, high, critical
target_object
string
нет
Фильтр по объекту воздействия


Response 200:

{

    "items": [

        {

            "rec_code": "REC-BIOS-001",

            "title": "Обновить прошивки BIOS/UEFI на всех хостах",

            "description": "Проверить наличие обновлений BIOS...",

            "priority": 1,

            "threat_code": "190",

            "threat_name": "Угроза внедрения вредоносного кода в BIOS",

            "target_object": "Микропрограммное обеспечение BIOS/UEFI",

            "vuln_level": "critical",

            "source_standard": "ФСТЭК, Приказ №239"

        }

    ],

    "total": 45

}


ENDPOINT 10: Demo Scenarios List
GET /api/v1/scenarios/demo

Описание: Список предзаготовленных сценариев для демонстрации на защите.

Response 200:

{

    "scenarios": [

        {

            "id": 1,

            "name": "Критический риск: Медицина, Якутия",

            "description": "Крупная больница с 1500 хостами в регионе с высокой активностью атак",

            "expected_risk": "critical",

            "request_preview": {

                "enterprise_type": "Медицина",

                "host_count": 1500,

                "region": "Республика Саха (Якутия)"

            }

        },

        {

            "id": 2,

            "name": "Средний риск: НКО, Москва",

            "description": "Небольшая некоммерческая организация в столице",

            "expected_risk": "medium",

            "request_preview": {

                "enterprise_type": "НКО",

                "host_count": 120,

                "region": "Москва"

            }

        },

        {

            "id": 3,

            "name": "Низкий риск: Образование, Краснодар",

            "description": "Учебное заведение с минимальной инфраструктурой",

            "expected_risk": "low",

            "request_preview": {

                "enterprise_type": "Образование",

                "host_count": 45,

                "region": "Краснодарский край"

            }

        }

    ]

}


ENDPOINT 11: Run Demo Scenario
POST /api/v1/scenarios/demo/{id}/run

Описание: Запуск демо-сценария. Берет request_payload из БД и прогоняет через /predict.

Path Parameters:

Параметр
Тип
Описание
id
int
ID сценария из таблицы demo_scenarios


Response 200: Идентичен ответу /api/v1/predict (та же структура JSON).

Response 404:

{

    "error": {

        "code": "SCENARIO_NOT_FOUND",

        "message": "Демо-сценарий с id=99 не найден"

    }

}

# ТЗ ДЛЯ BACKEND-РАЗРАБОТЧИКА

4.1. Зона ответственности
Ты отвечаешь за:

FastAPI приложение со всеми 11 endpoints
Подключение к PostgreSQL через SQLAlchemy
Валидацию всех входных данных через Pydantic
Интеграцию ML-моделей (загрузка .pkl, вызов predict)
Сборку финального JSON-ответа (обогащение данными из БД)
Сохранение каждого прогноза в таблицу predictions
Swagger документацию (автоматическая через FastAPI)
Обработку ошибок (все ответы по единому формату)
4.2. Структура кода
backend/src/

├── main.py                          # Точка входа FastAPI

├── api/

│   ├── deps.py                      # Зависимости (DB session, ML loader)

│   └── routes/

│       ├── health.py                # GET /health

│       ├── analytics.py             # GET /analytics/*

│       ├── threats.py               # GET /threats, GET /threats/{code}

│       ├── predict.py               # POST /predict

│       ├── recommendations.py       # GET /recommendations

│       └── scenarios.py             # GET+POST /scenarios/demo

├── core/

│   ├── config.py                    # Settings из .env

│   ├── logger.py                    # Логирование

│   └── exceptions.py               # Кастомные исключения

├── db/

│   ├── session.py                   # SQLAlchemy engine + session

│   ├── models.py                    # ORM-модели (зеркало SQL таблиц)

│   └── repositories/

│       ├── incidents.py             # CRUD + агрегации инцидентов

│       ├── threats.py               # CRUD угроз

│       ├── predictions.py           # Сохранение прогнозов

│       └── recommendations.py       # CRUD рекомендаций

├── schemas/

│   ├── analytics.py                 # Pydantic: AnalyticsSummary, Timeseries

│   ├── predict.py                   # Pydantic: PredictRequest, PredictResponse

│   ├── threat.py                    # Pydantic: ThreatItem, ThreatDetail

│   └── recommendation.py           # Pydantic: RecommendationItem

├── services/

│   ├── analytics_service.py         # Бизнес-логика аналитики

│   ├── prediction_service.py        # Оркестрация 6 моделей

│   └── recommendation_service.py    # Подбор рекомендаций по контексту

└── ml/

    ├── loader.py                    # Загрузка .pkl при старте

    ├── feature_builder.py           # Преобразование request → features

    ├── inference.py                 # Вызов моделей

    └── explainability.py            # SHAP-объяснения
4.3. Порядок работы по часам
Скелет
# main.py — первый коммит

from fastapi import FastAPI

from fastapi.middleware.cors import CORSMiddleware

app = FastAPI(

    title="Cyber Threat Predictor API",

    version="1.0.0",

    description="API системы прогнозирования киберугроз"

)

app.add_middleware(

    CORSMiddleware,

    allow_origins=["http://localhost:3000"],

    allow_methods=["*"],

    allow_headers=["*"],

)
Часы 2-4: Моки для фронтенда
# api/routes/predict.py — ЗАГЛУШКА (фронт уже может работать)

@router.post("/api/v1/predict")

async def predict_mock(data: dict):

    return {

        "request_id": "mock-001",

        "model_version": "v1",

        "inference_time_ms": 0,

        "incident_prediction": {

            "will_happen": True,

            "probability": 0.85,

            "confidence_level": "high",

            "confidence_label": "Высокая уверенность"

        },

        "attack_time_prediction": {

            "time_bucket": "night",

            "time_bucket_label": "Ночь (00:00 — 06:00)",

            "probable_hour": 2,

            "probable_day_of_week": "friday",

            "probable_day_label": "Пятница",

            "probable_season": "summer"

        },

        "threat_prediction": {

            "primary": {

                "threat_code": "190",

                "threat_name": "Угроза внедрения вредоносного кода в BIOS",

                "probability": 0.52

            },

            "top_3": []

        },

        "target_object_prediction": {

            "primary": {

                "object_code": "bios_firmware",

                "object_name": "Микропрограммное обеспечение BIOS/UEFI",

                "probability": 0.62

            },

            "top_3": []

        },

        "vulnerability_assessment": {

            "level": "critical",

            "level_label": "Критический",

            "score": 0.87,

            "factors": ["Мок-данные"]

        },

        "recommendations": [

            {

                "rec_code": "REC-001",

                "title": "Обновить прошивки BIOS",

                "description": "Мок-рекомендация",

                "priority": 1,

                "priority_label": "Критический",

                "related_threat": "190"

            }

        ],

        "explanations": {

            "method": "SHAP",

            "top_features": [

                {

                    "feature": "enterprise_type",

                    "display_name": "Отрасль: Медицина",

                    "impact": 0.21,

                    "direction": "increases_risk",

                    "explanation": "Мок-объяснение"

                }

            ]

        },

        "business_impact": {

            "estimated_damage_rub": 15200000,

            "damage_label": "~15.2 млн руб."

        }

    }
 БД + реальная аналитика
Подключить SQLAlchemy к PostgreSQL
Написать ORM-модели (зеркало SQL)
Реализовать repositories для incidents, threats
Реализовать analytics endpoints с реальными SQL-запросами
Часы 12-24: Реальная интеграция ML
Написать ml/loader.py (загрузка .pkl при старте)
Написать ml/feature_builder.py (request → DataFrame)
Написать ml/inference.py (вызов всех 6 моделей)
Заменить моки на реальные вызовы
Часы 24-36: Обогащение и рекомендации
Обогащение ответа данными из БД (threat_name, impact_object)
Сервис рекомендаций (по threat_code → рекомендации из БД)
SHAP-объяснения
Сохранение прогнозов в predictions
Часы 36-48: Полировка
Обработка всех ошибок
Логирование
Демо-сценарии
Финальное тестирование
4.4. Pydantic-схемы (ключевые)
# schemas/predict.py

from pydantic import BaseModel, Field

from typing import Optional, List

from enum import Enum

class EnterpriseType(str, Enum):

    MEDICINE = "Медицина"

    NGO = "НКО"

    EDUCATION = "Образование"

    INDUSTRY = "Промышленность"

    GOVERNMENT = "Госуправление"

    FINANCE = "Финансы"

    TRANSPORT = "Транспорт"

    ENERGY = "Энергетика"

    TELECOM = "Телеком"

    OTHER = "Другое"

class AdditionalContext(BaseModel):

    internet_exposed: Optional[bool] = None

    has_critical_data: Optional[bool] = None

    segmentation_level: Optional[str] = None

class PredictRequest(BaseModel):

    enterprise_type: EnterpriseType

    host_count: int = Field(gt=0, le=100000)

    region: str = Field(min_length=2, max_length=200)

    observation_date: str = Field(

        pattern=r"^\d{4}-\d{2}-\d{2}$"

    )

    additional_context: Optional[AdditionalContext] = None

class IncidentPrediction(BaseModel):

    will_happen: bool

    probability: float

    confidence_level: str

    confidence_label: str

class ThreatItem(BaseModel):

    threat_code: str

    threat_name: str

    probability: float

class ThreatPrediction(BaseModel):

    primary: ThreatItem

    top_3: List[ThreatItem]

class FeatureExplanation(BaseModel):

    feature: str

    display_name: str

    impact: float

    direction: str

    explanation: str

class RecommendationItem(BaseModel):

    rec_code: str

    title: str

    description: str

    priority: int

    priority_label: str

    related_threat: Optional[str]

class PredictResponse(BaseModel):

    request_id: str

    model_version: str

    inference_time_ms: int

    incident_prediction: IncidentPrediction

    attack_time_prediction: dict

    threat_prediction: ThreatPrediction

    target_object_prediction: dict

    vulnerability_assessment: dict

    recommendations: List[RecommendationItem]

    explanations: dict

    business_impact: dict
4.5. Нефункциональные требования
Требование
Значение
Время ответа /predict
< 300ms
Время ответа /analytics/*
< 100ms
Формат ошибок
Единый JSON с code + message + details
Swagger
Автоматический на /docs
CORS
Разрешен localhost:3000
Логирование
Каждый запрос + каждая ошибка
Type Hints
Обязательны на всех функциях
Форматирование
Black + Ruff перед финальным коммитом

# ТЗ ДЛЯ FRONTEND-РАЗРАБОТЧИКА


5.1. Зона ответственности
Ты отвечаешь за:

React-приложение с 5 экранами
Типизацию всех API-ответов (TypeScript)
Визуализацию графиков (Recharts)
Формы ввода с валидацией
Состояния loading / error / empty
Кнопку «Демо-сценарий» для защиты
Адаптивную верстку
Красивый UX для жюри
5.2. Структура кода
frontend/src/

├── main.tsx                         # Точка входа

├── app/

│   ├── router.tsx                   # React Router: 5 маршрутов

│   └── providers.tsx                # RTK Query provider, Theme

├── components/

│   ├── layout/

│   │   ├── Header.tsx               # Навигация

│   │   ├── Sidebar.tsx              # Боковое меню

│   │   └── PageWrapper.tsx          # Обертка страницы

│   ├── charts/

│   │   ├── IncidentTimeline.tsx     # Линейный график инцидентов

│   │   ├── RegionBarChart.tsx       # Горизонтальный bar по регионам

│   │   ├── ThreatPieChart.tsx       # Круговая диаграмма угроз

│   │   ├── EnterpriseTypeChart.tsx  # Bar по отраслям

│   │   └── ShapWaterfall.tsx        # Waterfall-диаграмма SHAP

│   ├── cards/

│   │   ├── KpiCard.tsx              # Карточка KPI (число + подпись)

│   │   ├── RiskCard.tsx             # Карточка уровня риска (цветная)

│   │   ├── ThreatCard.tsx           # Карточка угрозы

│   │   └── RecommendationCard.tsx   # Карточка рекомендации

│   ├── forms/

│   │   ├── PredictionForm.tsx       # Форма прогноза

│   │   └── FilterPanel.tsx          # Фильтры для дашборда

│   └── common/

│       ├── LoadingSpinner.tsx

│       ├── ErrorMessage.tsx

│       ├── EmptyState.tsx

│       └── Toast.tsx                # Уведомления

├── pages/

│   ├── DashboardPage.tsx            # Экран 1: Обзорная аналитика

│   ├── PredictionPage.tsx           # Экран 2: Прогнозирование

│   ├── VulnerabilityPage.tsx        # Экран 3: Диагностика уязвимостей

│   ├── RecommendationsPage.tsx      # Экран 4: Рекомендации

│   └── ThreatCatalogPage.tsx        # Экран 5: Каталог угроз ФСТЭК

├── services/

│   ├── api.ts                       # Базовый axios/fetch клиент

│   ├── analyticsApi.ts              # Вызовы /analytics/*

│   ├── predictApi.ts                # Вызов /predict

│   ├── threatsApi.ts                # Вызовы /threats

│   ├── recommendationsApi.ts        # Вызовы /recommendations

│   └── scenariosApi.ts              # Вызовы /scenarios/demo

├── types/

│   ├── api.ts                       # Общие типы (Error, Pagination)

│   ├── analytics.ts                 # Типы для аналитики

│   ├── prediction.ts                # Типы для прогноза

│   ├── threat.ts                    # Типы для угроз

│   └── recommendation.ts           # Типы для рекомендаций

├── hooks/

│   ├── useAnalytics.ts

│   ├── usePrediction.ts

│   └── useDemo.ts

├── utils/

│   ├── formatters.ts                # Форматирование чисел, дат

│   ├── riskColors.ts                # Цвета по уровню риска

│   └── constants.ts                 # Константы (URL API, лейблы)

└── assets/

    └── styles/

        └── globals.css
5.3. Экраны — детальное описание
Экран 1: Dashboard (Обзорная аналитика)
┌──────────────────────────────────────────────────────────────┐

│  HEADER: Система прогнозирования киберугроз                   │

├──────────────────────────────────────────────────────────────┤

│                                                               │

│  ┌─────────┐ ┌─────────┐ ┌─────────┐ ┌─────────┐            │

│  │  2000   │ │   943   │ │   187   │ │   89    │            │

│  │Инцидентов│ │Успешных │ │ Угроз   │ │Регионов │            │

│  │  всего  │ │  атак   │ │уникальных│ │         │            │

│  └─────────┘ └─────────┘ └─────────┘ └─────────┘            │

│                                                               │

│  ┌─── Фильтры ──────────────────────────────────────────┐    │

│  │ Регион: [▼ Все]  Отрасль: [▼ Все]  Период: [▼ Год]  │    │

│  └──────────────────────────────────────────────────────┘    │

│                                                               │

│  ┌─────────────────────────┐ ┌──────────────────────────┐    │

│  │  Инциденты по месяцам   │ │  Топ-10 регионов         │    │

│  │  (линейный график)      │ │  (горизонтальный bar)    │    │

│  │                         │ │                          │    │

│  │  ~~~~~/\~~~~/\~~~~      │ │  Хабаровский ████████   │    │

│  │  ~~~~/  \~~/  \~~~     │ │  Москва      ███████    │    │

│  │                         │ │  Якутия      ██████     │    │

│  └─────────────────────────┘ └──────────────────────────┘    │

│                                                               │

│  ┌─────────────────────────┐ ┌──────────────────────────┐    │

│  │  Топ-5 угроз            │ │  По отраслям             │    │

│  │  (pie chart)            │ │  (bar chart)             │    │

│  │                         │ │                          │    │

│  │      ╭───╮              │ │  Медицина  ████████████  │    │

│  │    ╭─┤   ├─╮            │ │  НКО       ████████     │    │

│  │    ╰─┤   ├─╯            │ │  Образов.  ██████       │    │

│  │      ╰───╯              │ │                          │    │

│  └─────────────────────────┘ └──────────────────────────┘    │

└──────────────────────────────────────────────────────────────┘

Источники данных:

KPI карточки → GET /api/v1/analytics/summary
График по месяцам → GET /api/v1/analytics/timeseries
Топ регионов → GET /api/v1/analytics/regions
По отраслям → GET /api/v1/analytics/enterprise-types


Экран 2: Prediction (Прогнозирование)
┌──────────────────────────────────────────────────────────────┐

│  ПРОГНОЗИРОВАНИЕ КИБЕРУГРОЗ                                   │

├──────────────────────────────────────────────────────────────┤

│                                                               │

│  ┌─── Форма ввода ──────────────────────────────────────┐    │

│  │                                                       │    │

│  │  Отрасль:    [▼ Медицина        ]                     │    │

│  │  Хосты:      [  1500            ]                     │    │

│  │  Регион:     [▼ Якутия          ]                     │    │

│  │  Дата:       [  2025-05-20      ]                     │    │

│  │                                                       │    │

│  │  [🎯 ДЕМО: Критический]  [🎯 ДЕМО: Средний]          │    │

│  │                                                       │    │

│  │         [ 🔍 АНАЛИЗИРОВАТЬ ]                          │    │

│  └───────────────────────────────────────────────────────┘    │

│                                                               │

│  ═══════════════ РЕЗУЛЬТАТЫ ═══════════════                   │

│                                                               │

│  ┌──────────────────┐  ┌──────────────────┐                  │

│  │  🔴 ВЕРОЯТНОСТЬ   │  │  🕐 ВРЕМЯ АТАКИ  │                  │

│  │     АТАКИ         │  │                  │                  │

│  │    ██████ 85%     │  │  Ночь 00-06      │                  │

│  │  Высокая уверен.  │  │  Пятница         │                  │

│  └──────────────────┘  └──────────────────┘                  │

│                                                               │

│  ┌──────────────────┐  ┌──────────────────┐                  │

│  │  ⚠️ ТИП УГРОЗЫ   │  │  🎯 ОБЪЕКТ       │                  │

│  │                  │  │                  │                  │

│  │  1. УБИ-190 52%  │  │  1. BIOS    62%  │                  │

│  │  2. УБИ-152 19%  │  │  2. Сервер  21%  │                  │

│  │  3. УБИ-163 11%  │  │  3. Сеть     9%  │                  │

│  └──────────────────┘  └──────────────────┘                  │

│                                                               │

│  ┌─── УРОВЕНЬ УЯЗВИМОСТИ ───────────────────────────────┐    │

│  │  🔴 КРИТИЧЕСКИЙ (0.87)                                │    │

│  │  ████████████████████████████░░░░                      │    │

│  │                                                       │    │

│  │  Факторы:                                             │    │

│  │  • Высокая вероятность успешной атаки (85%)           │    │

│  │  • Большое количество хостов (1500)                   │    │

│  │  • Регион с повышенной активностью                    │    │

│  └───────────────────────────────────────────────────────┘    │

│                                                               │

│  ┌─── ПОЧЕМУ ТАКОЙ ПРОГНОЗ (SHAP) ─────────────────────┐    │

│  │                                                       │    │

│  │  Отрасль: Медицина        ████████████  +21%          │    │

│  │  Хосты: 1500              █████████     +18%          │    │

│  │  Регион: Якутия           ██████        +12%          │    │

│  │  Сезон: Лето              ████          +8%           │    │

│  │  Сегментация: Низкая      ███           +7%           │    │

│  └───────────────────────────────────────────────────────┘    │

│                                                               │

│  ┌─── РЕКОМЕНДАЦИИ ─────────────────────────────────────┐    │

│  │                                                       │    │

│  │  🔴 1. Обновить прошивки BIOS/UEFI                    │    │

│  │     Связано с: УБИ-190                                │    │

│  │                                                       │    │

│  │  🟡 2. Усилить сегментацию сети                       │    │

│  │     Связано с: УБИ-190                                │    │

│  │                                                       │    │

│  │  🟢 3. Усилить ночной мониторинг (00:00-06:00)       │    │

│  │     На основе прогноза времени атаки                  │    │

│  └───────────────────────────────────────────────────────┘    │

│                                                               │

│  ┌─── БИЗНЕС-ЦЕННОСТЬ ─────────────────────────────────┐    │

│  │  💰 Потенциальный предотвращенный ущерб: ~15.2 млн ₽ │    │

│  └───────────────────────────────────────────────────────┘    │

└──────────────────────────────────────────────────────────────┘

Источник данных: POST /api/v1/predict

Кнопки ДЕМО: При нажатии заполняют форму предзаготовленными данными из GET /api/v1/scenarios/demo


Экран 3: Vulnerability (Диагностика уязвимостей)
┌──────────────────────────────────────────────────────────────┐

│  ДИАГНОСТИКА УЯЗВИМОСТЕЙ                                      │

├──────────────────────────────────────────────────────────────┤

│                                                               │

│  ┌─── Таблица предприятий ──────────────────────────────┐    │

│  │                                                       │    │

│  │  Код  │ Отрасль    │ Регион     │ Хосты │ Риск       │    │

│  │  ─────┼────────────┼────────────┼───────┼──────────  │    │

│  │  1825 │ Медицина   │ Якутия     │ 1500  │ 🔴 0.87   │    │

│  │  1340 │ НКО        │ Москва     │  120  │ 🟡 0.45   │    │

│  │  2001 │ Образование│ Краснодар  │   45  │ 🟢 0.21   │    │

│  │                                                       │    │

│  │  Сортировка: [▼ По риску ↓]  Фильтр: [▼ Все отрасли] │    │

│  └───────────────────────────────────────────────────────┘    │

│                                                               │

│  ┌─── Детали выбранного предприятия ────────────────────┐    │

│  │                                                       │    │

│  │  Предприятие: 1825 (Медицина, Якутия)                │    │

│  │  Всего инцидентов: 12  │  Успешных: 8 (67%)          │    │

│  │  Последний инцидент: 2025-04-15                       │    │

│  │                                                       │    │

│  │  Факторы риска:                                       │    │

│  │  • Высокая доля успешных атак                         │    │

│  │  • Большая поверхность атаки (1500 хостов)           │    │

│  │  • Регион с повышенной активностью                    │    │

│  └───────────────────────────────────────────────────────┘    │

└──────────────────────────────────────────────────────────────┘

Источники данных:

Таблица → GET /api/v1/analytics/summary + данные из enterprise_profiles
Детали → комбинация аналитики и прогноза


Экран 4: Recommendations (Рекомендации)
┌──────────────────────────────────────────────────────────────┐

│  РЕКОМЕНДАЦИИ ПО ЗАЩИТЕ                                       │

├──────────────────────────────────────────────────────────────┤

│                                                               │

│  Фильтры: Угроза [▼ Все]  Уровень [▼ Все]  Объект [▼ Все]  │

│                                                               │

│  ┌─── Приоритет 1 (Критический) ────────────────────────┐    │

│  │  🔴 REC-BIOS-001                                      │    │

│  │  Обновить прошивки BIOS/UEFI на всех хостах           │    │

│  │  Угроза: УБИ-190 │ Объект: BIOS │ Стандарт: ФСТЭК    │    │

│  └───────────────────────────────────────────────────────┘    │

│                                                               │

│  ┌─── Приоритет 2 (Высокий) ───────────────────────────┐    │

│  │  🟡 REC-NET-003                                       │    │

│  │  Усилить сегментацию сети                             │    │

│  │  Угроза: УБИ-190 │ Объект: Сеть │ Стандарт: ФСТЭК    │    │

│  └───────────────────────────────────────────────────────┘    │

└──────────────────────────────────────────────────────────────┘

Источник данных: GET /api/v1/recommendations


Экран 5: Threat Catalog (Каталог угроз)
┌──────────────────────────────────────────────────────────────┐

│  КАТАЛОГ УГРОЗ ФСТЭК                                         │

├──────────────────────────────────────────────────────────────┤

│                                                               │

│  Поиск: [  вредоносный код                    ] 🔍           │

│                                                               │

│  ┌─── УБИ-001 ─────────────────────────────────────────┐    │

│  │  Угроза автоматического распространения              │    │

│  │  вредоносного кода в грид-системе                    │    │

│  │                                                       │    │

│  │  Объект: Ресурсные центры грид-системы               │    │

│  │  Нарушения: 🔒К  🔧Ц  ⚡Д                            │    │

│  │  Инцидентов в базе: 12                                │    │

│  │  Статус: Опубликована                                 │    │

│  └───────────────────────────────────────────────────────┘    │

│                                                               │

│  ┌─── УБИ-002 ─────────────────────────────────────────┐    │

│  │  Угроза агрегирования данных, передаваемых           │    │

│  │  датчиками                                            │    │

│  │  ...                                                  │    │

│  └───────────────────────────────────────────────────────┘    │

│                                                               │

│  Страница: [1] 2 3 4 ... 12                                  │

└──────────────────────────────────────────────────────────────┘

Источник данных: GET /api/v1/threats и GET /api/v1/threats/{ubi_code}
5.4. TypeScript типы (ключевые)
// types/prediction.ts

export interface PredictRequest {

    enterprise_type: EnterpriseType;

    host_count: number;

    region: string;

    observation_date: string;

    additional_context?: {

        internet_exposed?: boolean;

        has_critical_data?: boolean;

        segmentation_level?: 'low' | 'medium' | 'high';

    };

}

export type EnterpriseType =

    | 'Медицина'

    | 'НКО'

    | 'Образование'

    | 'Промышленность'

    | 'Госуправление'

    | 'Финансы'

    | 'Транспорт'

    | 'Энергетика'

    | 'Телеком'

    | 'Другое';

export interface PredictResponse {

    request_id: string;

    model_version: string;

    inference_time_ms: number;

    incident_prediction: IncidentPrediction;

    attack_time_prediction: AttackTimePrediction;

    threat_prediction: ThreatPrediction;

    target_object_prediction: TargetObjectPrediction;

    vulnerability_assessment: VulnerabilityAssessment;

    recommendations: RecommendationItem[];

    explanations: Explanations;

    business_impact: BusinessImpact;

}

export interface IncidentPrediction {

    will_happen: boolean;

    probability: number;

    confidence_level: 'low' | 'medium' | 'high';

    confidence_label: string;

}

export interface AttackTimePrediction {

    time_bucket: 'night' | 'morning' | 'day' | 'evening';

    time_bucket_label: string;

    probable_hour: number;

    probable_day_of_week: string;

    probable_day_label: string;

    probable_season: string;

}

export interface ThreatItem {

    threat_code: string;

    threat_name: string;

    probability: number;

}

export interface ThreatPrediction {

    primary: ThreatItem;

    top_3: ThreatItem[];

}

export interface TargetObjectItem {

    object_code: string;

    object_name: string;

    probability: number;

}

export interface TargetObjectPrediction {

    primary: TargetObjectItem;

    top_3: TargetObjectItem[];

}

export interface VulnerabilityAssessment {

    level: 'low' | 'medium' | 'high' | 'critical';

    level_label: string;

    score: number;

    factors: string[];

}

export interface FeatureExplanation {

    feature: string;

    display_name: string;

    impact: number;

    direction: 'increases_risk' | 'decreases_risk';

    explanation: string;

}

export interface Explanations {

    method: string;

    top_features: FeatureExplanation[];

}

export interface RecommendationItem {

    rec_code: string;

    title: string;

    description: string;

    priority: number;

    priority_label: string;

    related_threat: string | null;

}

export interface BusinessImpact {

    estimated_damage_rub: number;

    damage_label: string;

    calculation_basis?: string;

}

// types/analytics.ts

export interface AnalyticsSummary {

    totals: {

        total_incidents: number;

        successful_incidents: number;

        success_rate: number;

        unique_threats: number;

        unique_regions: number;

        unique_enterprises: number;

        avg_host_count: number;

    };

    top_threats: Array<{

        threat_code: string;

        threat_name: string;

        count: number;

        success_rate: number;

    }>;

    top_regions: Array<{

        region: string;

        count: number;

        success_rate: number;

    }>;

    top_enterprise_types: Array<{

        enterprise_type: string;

        count: number;

        success_rate: number;

    }>;

    filters_applied: Record<string, string | null>;

}

export interface TimeseriesPoint {

    period: string;

    total: number;

    successful: number;

    failed: number;

}

export interface TimeseriesResponse {

    granularity: string;

    series: TimeseriesPoint[];

}
5.5. Цветовая схема рисков
// utils/riskColors.ts

export const RISK_COLORS = {

    critical: { bg: '#FEE2E2', text: '#991B1B', border: '#EF4444', icon: '🔴' },

    high:     { bg: '#FEF3C7', text: '#92400E', border: '#F59E0B', icon: '🟠' },

    medium:   { bg: '#FEF9C3', text: '#854D0E', border: '#EAB308', icon: '🟡' },

    low:      { bg: '#DCFCE7', text: '#166534', border: '#22C55E', icon: '🟢' },

} as const;

export function getRiskColor(level: string) {

    return RISK_COLORS[level as keyof typeof RISK_COLORS]

        ?? RISK_COLORS.medium;

}

export function getRiskLevel(score: number): string {

    if (score >= 0.75) return 'critical';

    if (score >= 0.50) return 'high';

    if (score >= 0.25) return 'medium';

    return 'low';

}
5.6. Порядок работы по часам
Часы 0-4: Скелет
Vite + React + TypeScript + React Router
Layout (Header + Sidebar + PageWrapper)
5 пустых страниц с маршрутами
Базовый API-клиент (axios с baseURL)
Часы 4-12: Dashboard на моках
KPI карточки (компонент KpiCard)
Графики (Recharts: LineChart, BarChart, PieChart)
Фильтры (FilterPanel)
Подключение к mock-endpoints бэкенда
Часы 12-24: Prediction Page
PredictionForm (выпадающие списки, валидация)
Блоки результатов (RiskCard, ThreatCard)
SHAP waterfall (ShapWaterfall)
Рекомендации (RecommendationCard)
Кнопки демо-сценариев
Часы 24-36: Остальные экраны
VulnerabilityPage (таблица + детали)
RecommendationsPage (список + фильтры)
ThreatCatalogPage (поиск + пагинация + карточки)
Часы 36-48: Полировка
Loading/Error/Empty states на всех экранах
Переключение на реальный API
Кнопка «Демо-сценарий» работает в один клик
Адаптивность
Toast-уведомления при ошибках

# ТЗ ДЛЯ ML-ИНЖЕНЕРА

# ТЗ ДЛЯ ML-ИНЖЕНЕРА
6.1. Зона ответственности
Ты отвечаешь за:

Очистку и канонизацию данных из raw файлов
Feature engineering (создание признаков)
Обучение 6 моделей
Оценку качества (метрики)
Экспорт артефактов (.pkl + metadata)
SHAP-объяснения
Передачу backend-ready моделей с документацией
6.2. Структура кода
ml/src/

├── data/

│   ├── loaders.py              # Чтение xlsx/csv

│   ├── cleaning.py             # Очистка, дедупликация

│   └── validation.py           # Проверка целостности

├── features/

│   ├── base.py                 # Базовые признаки

│   ├── temporal.py             # Временные (hour, season, day_of_week)

│   ├── categorical.py          # Категориальные (enterprise_type, region)

│   └── aggregations.py         # Агрегаты (incident_count_by_region и т.д.)

├── models/

│   ├── incident.py             # Модель 1: Факт инцидента

│   ├── attack_time.py          # Модель 2: Время атаки

│   ├── threat_type.py          # Модель 3: Тип угрозы

│   ├── target_object.py        # Модель 4: Объект воздействия

│   ├── vulnerability.py        # Модель 5: Уровень уязвимости

│   └── recommendation.py       # Модель 6: Рекомендации

├── pipelines/

│   ├── train.py                # Обучение всех моделей

│   ├── evaluate.py             # Оценка метрик

│   └── export.py               # Экспорт .pkl + metadata

└── utils/

    ├── metrics.py              # Расчет метрик

    └── shap_utils.py           # SHAP-объяснения
6.3. Этап 1: Очистка данных (Часы 0-8)
Проблемы в сырых данных:
Дубликаты предприятий с разными host_count и регионами
Неконсистентные форматы дат
Разные написания регионов («Москва» vs «г. Москва» vs «Московская обл.»)
Разные написания типов предприятий
Пропущенные значения
Правила канонизации:
# scripts/canonicalize_data.py

REGION_MAPPING = {

    "г. Москва": "Москва",

    "Московская обл.": "Московская область",

    "Респ. Саха": "Республика Саха (Якутия)",

    "Респ. Марий": "Республика Марий Эл",

    # ... полный маппинг

}

ENTERPRISE_TYPE_MAPPING = {

    "медицина": "Медицина",

    "Мед.": "Медицина",

    "нко": "НКО",

    "некоммерческая": "НКО",

    # ... полный маппинг

}

def resolve_host_conflict(group):

    """При конфликте host_count для одного enterprise_code

    берем максимальное значение (консервативная оценка)"""

    return group['host_count'].max()

def resolve_region_conflict(group):

    """При конфликте региона берем самый частый"""

    return group['region'].mode().iloc[0]
Результат очистки:
data/processed/incidents_clean.parquet
data/processed/threats_clean.parquet
data/processed/enterprise_profiles.parquet
docs/DATA_CLEANING.md — описание всех правил
6.4. Этап 2: Feature Engineering (Часы 8-16)
Полный список признаков:
# features/base.py — Базовые из сырых данных

BASE_FEATURES = [

    'enterprise_type',      # Категория: Медицина, НКО, ...

    'host_count',           # Число: 1-100000

    'region',               # Категория: 89 регионов

    'threat_code',          # Категория: код угрозы

]

# features/temporal.py — Производные временные

TEMPORAL_FEATURES = [

    'incident_hour',        # 0-23

    'hour_sin',             # sin(2π * hour/24) — циклическое кодирование

    'hour_cos',             # cos(2π * hour/24)

    'day_of_week',          # 0-6 (пн-вс)

    'is_weekend',           # 0 или 1

    'month',                # 1-12

    'month_sin',            # sin(2π * month/12)

    'month_cos',            # cos(2π * month/12)

    'quarter',              # 1-4

    'season',               # winter/spring/summer/autumn

    'time_bucket',          # night(0-6)/morning(6-12)/day(12-18)/evening(18-24)

]

# features/aggregations.py — Агрегированные (считаются по всему датасету)

AGGREGATED_FEATURES = [

    'incidents_by_enterprise',      # Сколько инцидентов у этого предприятия

    'success_rate_by_enterprise',   # Доля успешных атак у предприятия

    'incidents_by_region',          # Сколько инцидентов в регионе

    'success_rate_by_region',       # Доля успешных атак в регионе

    'incidents_by_type',            # Сколько инцидентов в отрасли

    'success_rate_by_type',         # Доля успешных атак в отрасли

    'avg_hosts_by_type',            # Средний host_count в отрасли

    'threat_frequency',             # Как часто встречается эта угроза

    'days_since_last_incident',     # Давность последнего инцидента

]

# features/categorical.py — Из справочника угроз

THREAT_FEATURES = [

    'conf_violation',       # Нарушение конфиденциальности (0/1)

    'integ_violation',      # Нарушение целостности (0/1)

    'avail_violation',      # Нарушение доступности (0/1)

    'cia_sum',              # Сумма нарушений (0-3) — критичность

]
Циклическое кодирование времени:
import numpy as np

def encode_cyclical(value, max_value):

    """Час 23 и час 0 должны быть рядом, а не на разных концах шкалы"""

    sin_val = np.sin(2 * np.pi * value / max_value)

    cos_val = np.cos(2 * np.pi * value / max_value)

    return sin_val, cos_val
6.5. Этап 3: Обучение 6 моделей (Часы 16-32)
Модель 1: Факт инцидента (будет ли атака успешной)
# ml/src/models/incident.py

"""

Target: success (0/1)

Тип: Бинарная классификация

Алгоритм: CatBoostClassifier

Почему CatBoost: категориальные фичи (region, enterprise_type) без One-Hot

"""

from catboost import CatBoostClassifier

from sklearn.model_selection import StratifiedKFold

import shap

FEATURES = [

    'enterprise_type',          # cat

    'host_count',               # num

    'region',                   # cat

    'month_sin', 'month_cos',   # num

    'hour_sin', 'hour_cos',     # num

    'day_of_week',              # num

    'is_weekend',               # num

    'season',                   # cat

    'incidents_by_region',      # num

    'success_rate_by_region',   # num

    'incidents_by_type',        # num

    'success_rate_by_type',     # num

    'avg_hosts_by_type',        # num

    'threat_frequency',         # num

    'cia_sum',                  # num

]

CAT_FEATURES = ['enterprise_type', 'region', 'season']

TARGET = 'success'

def train_incident_model(df):

    X = df[FEATURES]

    y = df[TARGET]

    model = CatBoostClassifier(

        iterations=500,

        learning_rate=0.05,

        depth=6,

        l2_leaf_reg=3,

        cat_features=CAT_FEATURES,

        eval_metric='AUC',

        random_seed=42,

        verbose=100,

        early_stopping_rounds=50,

    )

    # 5-fold стратифицированная кросс-валидация

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_seed=42)

    scores = []

    for train_idx, val_idx in cv.split(X, y):

        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]

        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model.fit(

            X_train, y_train,

            eval_set=(X_val, y_val),

            verbose=False

        )

        scores.append(model.get_best_score()['validation']['AUC'])

    # Финальное обучение на всех данных

    model.fit(X, y, verbose=100)

    # SHAP explainer

    explainer = shap.TreeExplainer(model)

    return model, explainer, {

        'cv_auc_mean': round(sum(scores) / len(scores), 4),

        'cv_auc_std': round(

            (sum((s - sum(scores)/len(scores))**2 for s in scores)

             / len(scores)) ** 0.5, 4

        ),

        'cv_folds': 5,

    }

Метрики для отчета:

Метрика
Целевое значение
Описание
ROC-AUC
> 0.75
Качество разделения классов
F1
> 0.70
Баланс precision/recall
PR-AUC
> 0.65
Важно при дисбалансе классов



Модель 2: Время атаки
# ml/src/models/attack_time.py

"""

Target: time_bucket (night/morning/day/evening)

Тип: Мультиклассовая классификация

Алгоритм: CatBoostClassifier (multiclass)

Дополнительно: отдельная регрессия на hour

"""

from catboost import CatBoostClassifier, CatBoostRegressor

TIME_BUCKETS = {

    'night':   (0, 6),    # 00:00 — 05:59

    'morning': (6, 12),   # 06:00 — 11:59

    'day':     (12, 18),  # 12:00 — 17:59

    'evening': (18, 24),  # 18:00 — 23:59

}

FEATURES_TIME = [

    'enterprise_type',

    'host_count',

    'region',

    'month_sin', 'month_cos',

    'day_of_week',

    'is_weekend',

    'season',

    'incidents_by_region',

    'success_rate_by_type',

    'threat_frequency',

]

CAT_FEATURES_TIME = ['enterprise_type', 'region', 'season']

def train_time_bucket_model(df):

    """Классификация: в какой период суток произойдет атака"""

    X = df[FEATURES_TIME]

    y = df['time_bucket']

    model = CatBoostClassifier(

        iterations=400,

        learning_rate=0.05,

        depth=5,

        cat_features=CAT_FEATURES_TIME,

        eval_metric='MultiClass',

        random_seed=42,

        verbose=100,

    )

    model.fit(X, y)

    return model

def train_hour_model(df):

    """Регрессия: конкретный час (для более точного прогноза)"""

    X = df[FEATURES_TIME]

    y = df['incident_hour']

    model = CatBoostRegressor(

        iterations=300,

        learning_rate=0.05,

        depth=5,

        cat_features=CAT_FEATURES_TIME,

        eval_metric='MAE',

        random_seed=42,

        verbose=100,

    )

    model.fit(X, y)

    return model

Метрики:

Метрика
Целевое значение
Описание
Accuracy (time_bucket)
> 0.40
4 класса, random = 0.25
Macro-F1 (time_bucket)
> 0.35
Баланс по всем периодам
MAE (hour)
< 5 часов
Средняя ошибка в часах



Модель 3: Тип угрозы
# ml/src/models/threat_type.py

"""

Target: threat_code (мультикласс, ~50-100 уникальных кодов)

Тип: Мультиклассовая классификация

Алгоритм: LightGBM (быстрее CatBoost на большом числе классов)

Выход: top-3 вероятности

"""

import lightgbm as lgb

from sklearn.preprocessing import LabelEncoder

import numpy as np

FEATURES_THREAT = [

    'enterprise_type_encoded',

    'host_count',

    'region_encoded',

    'month_sin', 'month_cos',

    'hour_sin', 'hour_cos',

    'day_of_week',

    'is_weekend',

    'season_encoded',

    'incidents_by_region',

    'success_rate_by_region',

    'incidents_by_type',

    'avg_hosts_by_type',

]

def train_threat_model(df):

    # LightGBM требует числовое кодирование категорий

    le_enterprise = LabelEncoder()

    le_region = LabelEncoder()

    le_season = LabelEncoder()

    le_threat = LabelEncoder()

    df_encoded = df.copy()

    df_encoded['enterprise_type_encoded'] = le_enterprise.fit_transform(

        df['enterprise_type']

    )

    df_encoded['region_encoded'] = le_region.fit_transform(df['region'])

    df_encoded['season_encoded'] = le_season.fit_transform(df['season'])

    df_encoded['threat_code_encoded'] = le_threat.fit_transform(

        df['threat_code']

    )

    X = df_encoded[FEATURES_THREAT]

    y = df_encoded['threat_code_encoded']

    model = lgb.LGBMClassifier(

        n_estimators=500,

        learning_rate=0.05,

        max_depth=7,

        num_leaves=31,

        objective='multiclass',

        num_class=len(le_threat.classes_),

        random_state=42,

        verbose=-1,

    )

    model.fit(X, y)

    encoders = {

        'enterprise_type': le_enterprise,

        'region': le_region,

        'season': le_season,

        'threat': le_threat,

    }

    return model, encoders

def predict_top3_threats(model, encoders, features_dict):

    """Возвращает top-3 наиболее вероятных угрозы"""

    probs = model.predict_proba([list(features_dict.values())])[0]

    top3_idx = np.argsort(probs)[-3:][::-1]

    le_threat = encoders['threat']

    return [

        {

            'threat_code': str(le_threat.inverse_transform([idx])[0]),

            'probability': round(float(probs[idx]), 4),

        }

        for idx in top3_idx

    ]

Метрики:

Метрика
Целевое значение
Описание
Accuracy@1
> 0.20
Правильный код в топ-1
Accuracy@3
> 0.45
Правильный код в топ-3
Macro-F1
> 0.15
Сложно из-за числа классов



Модель 4: Объект воздействия
# ml/src/models/target_object.py

"""

Target: target_object (укрупненная категория)

Тип: Rule-based + классификация

КЛЮЧЕВОЙ ХАК: объект воздействия жестко привязан к типу угрозы

в справочнике ФСТЭК. Поэтому основной подход — маппинг.

"""

# Маппинг из thrlist.xlsx (impact_object → укрупненная категория)

OBJECT_MAPPING = {

    'Микропрограммное обеспечение BIOS/UEFI': 'bios_firmware',

    'Ресурсные центры грид-системы': 'server',

    'Сетевое программное обеспечение': 'network',

    'Системное программное обеспечение': 'workstation',

    'Средства защиты информации': 'security_tools',

    'Прикладное программное обеспечение': 'application',

    'Базы данных': 'database',

    'Виртуальная инфраструктура': 'cloud_resource',

    'Датчики, контроллеры': 'iot_device',

    # ... полный маппинг из thrlist.xlsx

}

OBJECT_LABELS = {

    'bios_firmware':   'Микропрограммное обеспечение BIOS/UEFI',

    'server':          'Серверное оборудование',

    'network':         'Сетевой сегмент',

    'workstation':     'Рабочие станции',

    'security_tools':  'Средства защиты информации',

    'application':     'Прикладное ПО',

    'database':        'Базы данных',

    'cloud_resource':  'Виртуальная инфраструктура',

    'iot_device':      'IoT-устройства',

    'other':           'Прочее',

}

def predict_object_by_threat(threat_code: str, threats_db: dict) -> dict:

    """

    Основной подход: берем predicted threat_code из Модели 3,

    находим impact_object в справочнике ФСТЭК,

    маппим на укрупненную категорию.

    Точность = 100% (детерминированный маппинг).

    """

    threat_info = threats_db.get(threat_code, {})

    raw_object = threat_info.get('impact_object', '')

    # Ищем совпадение в маппинге

    for pattern, code in OBJECT_MAPPING.items():

        if pattern.lower() in raw_object.lower():

            return {

                'object_code': code,

                'object_name': OBJECT_LABELS[code],

                'probability': 1.0,

                'method': 'rule_based',

            }

    return {

        'object_code': 'other',

        'object_name': OBJECT_LABELS['other'],

        'probability': 0.5,

        'method': 'fallback',

    }

def predict_object_top3(threat_top3: list, threats_db: dict) -> list:

    """

    Для top-3 объектов: берем top-3 угрозы из Модели 3,

    для каждой находим объект, возвращаем уникальные.

    """

    seen = set()

    result = []

    for threat in threat_top3:

        obj = predict_object_by_threat(threat['threat_code'], threats_db)

        if obj['object_code'] not in seen:

            seen.add(obj['object_code'])

            obj['probability'] = threat['probability']

            result.append(obj)

    return result[:3]

Почему rule-based, а не ML:

Объект воздействия детерминированно связан с типом угрозы в справочнике ФСТЭК
Обучать модель на то, что уже есть в справочнике — бессмысленно
100% точность маппинга vs ~30% точность модели на мультиклассе
Экономия 4-6 часов работы ML-инженера


Модель 5: Уровень уязвимости
# ml/src/models/vulnerability.py

"""

Target: vulnerability_level (low/medium/high/critical)

Тип: Scoring formula (эвристика)

КЛЮЧЕВОЙ ХАК: в данных НЕТ результатов сканеров уязвимостей.

Мы синтезируем proxy-метрику на основе имеющихся данных.

Это честно, если описано в документации.

"""

import numpy as np

def calculate_vulnerability_score(

    incident_probability: float,

    host_count: int,

    max_host_count: int,

    success_rate_by_type: float,

    success_rate_by_region: float,

    cia_sum: int,

    days_since_last_incident: int,

) -> dict:

    """

    Формула:

    Score = w1 * P(атака) +

            w2 * (hosts/max_hosts) +

            w3 * success_rate_type +

            w4 * success_rate_region +

            w5 * (cia_sum/3) +

            w6 * recency_factor

    Веса подобраны экспертно (можно обосновать на защите).

    """

    # Нормализация host_count

    host_ratio = min(host_count / max(max_host_count, 1), 1.0)

    # Фактор давности: чем недавнее инцидент, тем выше риск

    if days_since_last_incident <= 30:

        recency = 1.0

    elif days_since_last_incident <= 90:

        recency = 0.7

    elif days_since_last_incident <= 365:

        recency = 0.4

    else:

        recency = 0.1

    # Взвешенная сумма

    score = (

        0.30 * incident_probability +

        0.20 * host_ratio +

        0.15 * success_rate_by_type +

        0.15 * success_rate_by_region +

        0.10 * (cia_sum / 3.0) +

        0.10 * recency

    )

    score = round(np.clip(score, 0.0, 1.0), 4)

    # Определение уровня

    if score >= 0.75:

        level = 'critical'

        label = 'Критический'

    elif score >= 0.50:

        level = 'high'

        label = 'Высокий'

    elif score >= 0.25:

        level = 'medium'

        label = 'Средний'

    else:

        level = 'low'

        label = 'Низкий'

    # Формирование факторов (для UI)

    factors = []

    if incident_probability > 0.7:

        factors.append(

            f"Высокая вероятность успешной атаки "

            f"({int(incident_probability*100)}%)"

        )

    if host_ratio > 0.5:

        factors.append(

            f"Большое количество хостов ({host_count})"

        )

    if success_rate_by_region > 0.5:

        factors.append(

            "Регион с повышенной активностью атак"

        )

    if success_rate_by_type > 0.5:

        factors.append(

            "Отрасль в зоне повышенного риска"

        )

    if cia_sum == 3:

        factors.append(

            "Угроза нарушает конфиденциальность, "

            "целостность и доступность"

        )

    if recency > 0.7:

        factors.append(

            "Недавний инцидент повышает вероятность повторной атаки"

        )

    return {

        'level': level,

        'level_label': label,

        'score': score,

        'factors': factors,

    }

Обоснование для жюри:

«В исходных данных отсутствуют результаты сканирования уязвимостей. Мы построили proxy-метрику на основе 6 факторов с экспертными весами. Формула прозрачна и объяснима, что важнее черного ящика нейросети.»


Модель 6: Рекомендации
# ml/src/models/recommendation.py

"""

Target: рекомендуемые действия

Тип: Rule engine + scoring

Подход: НЕ чистая ML-модель, а гибридная система.

ML определяет контекст → правила выбирают рекомендации → скоринг ранжирует.

"""

from typing import List, Dict

# Справочник рекомендаций (загружается из БД при старте)

# Подготовлен заранее: 227 угроз × 2 рекомендации = ~450 записей

# Тексты сгенерированы через LLM до хакатона

def select_recommendations(

    predicted_threat_code: str,

    predicted_object: str,

    vuln_level: str,

    attack_time_bucket: str,

    recommendations_db: list,

    top_n: int = 3,

) -> List[Dict]:

    """

    Алгоритм:

    1. Фильтруем рекомендации по threat_code

    2. Если мало — добавляем по target_object

    3. Добавляем контекстные (по времени, уровню)

    4. Ранжируем по priority

    5. Возвращаем top_n

    """

    candidates = []

    # 1. По угрозе (самые релевантные)

    for rec in recommendations_db:

        if rec['threat_code'] == predicted_threat_code:

            rec['relevance_score'] = 1.0

            candidates.append(rec)

    # 2. По объекту воздействия

    for rec in recommendations_db:

        if (rec['target_object'] == predicted_object

                and rec not in candidates):

            rec['relevance_score'] = 0.7

            candidates.append(rec)

    # 3. По уровню уязвимости

    for rec in recommendations_db:

        if (rec.get('vuln_level') == vuln_level

                and rec not in candidates):

            rec['relevance_score'] = 0.5

            candidates.append(rec)

    # 4. Контекстная рекомендация по времени

    time_rec = generate_time_recommendation(attack_time_bucket)

    if time_rec:

        candidates.append(time_rec)

    # 5. Ранжирование: priority * relevance_score

    candidates.sort(

        key=lambda r: r['priority'] * r.get('relevance_score', 0.5),

        reverse=False  # меньший priority = выше приоритет

    )

    return candidates[:top_n]


def generate_time_recommendation(time_bucket: str) -> dict:

    """Контекстная рекомендация на основе прогноза времени"""

    labels = {

        'night':   'Усилить ночной мониторинг (00:00—06:00)',

        'morning': 'Усилить утренний мониторинг (06:00—12:00)',

        'day':     'Усилить дневной мониторинг (12:00—18:00)',

        'evening': 'Усилить вечерний мониторинг (18:00—00:00)',

    }

    descriptions = {

        'night': (

            'Модель прогнозирует атаку в ночное время. '

            'Рекомендуется усилить дежурную смену SOC '

            'и настроить автоматические алерты.'

        ),

        'morning': (

            'Модель прогнозирует атаку в утренние часы. '

            'Рекомендуется проверить логи до начала рабочего дня.'

        ),

        'day': (

            'Модель прогнозирует атаку в рабочее время. '

            'Рекомендуется усилить контроль пользовательских сессий.'

        ),

        'evening': (

            'Модель прогнозирует атаку в вечернее время. '

            'Рекомендуется настроить алерты на аномальную '

            'активность после рабочего дня.'

        ),

    }

    return {

        'rec_code': f'REC-TIME-{time_bucket.upper()}',

        'title': labels.get(time_bucket, 'Усилить мониторинг'),

        'description': descriptions.get(time_bucket, ''),

        'priority': 3,

        'priority_label': 'Средний',

        'related_threat': None,

        'relevance_score': 0.6,

    }


6.6. Этап 4: SHAP-объяснения (Часы 32-38)
# ml/src/utils/shap_utils.py

import shap

import numpy as np

from typing import List, Dict

FEATURE_DISPLAY_NAMES = {

    'enterprise_type': 'Отрасль',

    'host_count': 'Количество хостов',

    'region': 'Регион',

    'month_sin': 'Месяц (сезонность)',

    'month_cos': 'Месяц (сезонность)',

    'hour_sin': 'Час (цикличность)',

    'hour_cos': 'Час (цикличность)',

    'day_of_week': 'День недели',

    'is_weekend': 'Выходной день',

    'season': 'Сезон',

    'incidents_by_region': 'Инциденты в регионе',

    'success_rate_by_region': 'Успешность атак в регионе',

    'incidents_by_type': 'Инциденты в отрасли',

    'success_rate_by_type': 'Успешность атак в отрасли',

    'avg_hosts_by_type': 'Средние хосты в отрасли',

    'threat_frequency': 'Частота угрозы',

    'cia_sum': 'Критичность угрозы (КЦД)',

}

def get_shap_explanation(

    explainer: shap.TreeExplainer,

    features_dict: dict,

    feature_names: list,

    top_n: int = 5,

) -> List[Dict]:

    """

    Генерирует top-N объяснений в формате, готовом для UI.

    """

    import pandas as pd

    features_df = pd.DataFrame([features_dict])

    shap_values = explainer.shap_values(features_df)

    # Для бинарной классификации берем класс 1 (атака)

    if isinstance(shap_values, list):

        values = shap_values[1][0]

    else:

        values = shap_values[0]

    # Сортируем по абсолютному значению

    abs_values = np.abs(values)

    top_indices = np.argsort(abs_values)[-top_n:][::-1]

    explanations = []

    for idx in top_indices:

        feature_name = feature_names[idx]

        shap_value = float(values[idx])

        feature_value = features_dict[feature_name]

        # Формируем человекочитаемое объяснение

        display_name = FEATURE_DISPLAY_NAMES.get(

            feature_name, feature_name

        )

        if isinstance(feature_value, (int, float)):

            display_value = f"{display_name}: {feature_value}"

        else:

            display_value = f"{display_name}: {feature_value}"

        direction = 'increases_risk' if shap_value > 0 else 'decreases_risk'

        explanation_text = generate_explanation_text(

            feature_name, feature_value, shap_value

        )

        explanations.append({

            'feature': feature_name,

            'display_name': display_value,

            'impact': round(abs(shap_value), 4),

            'direction': direction,

            'explanation': explanation_text,

        })

    return explanations


def generate_explanation_text(

    feature_name: str,

    feature_value,

    shap_value: float,

) -> str:

    """Генерирует текстовое объяснение для конкретного признака"""

    templates = {

        'enterprise_type': {

            'positive': (

                f"Отрасль «{feature_value}» атакуется "

                f"чаще среднего по статистике"

            ),

            'negative': (

                f"Отрасль «{feature_value}» атакуется "

                f"реже среднего"

            ),

        },

        'host_count': {

            'positive': (

                f"Большая поверхность атаки "

                f"({feature_value} хостов) увеличивает вероятность"

            ),

            'negative': (

                f"Небольшое количество хостов "

                f"({feature_value}) снижает привлекательность цели"

            ),

        },

        'region': {

            'positive': (

                f"В регионе «{feature_value}» "

                f"зафиксирован рост инцидентов"

            ),

            'negative': (

                f"Регион «{feature_value}» "

                f"характеризуется низкой активностью атак"

            ),

        },

        'success_rate_by_region': {

            'positive': (

                "Высокая доля успешных атак в регионе "

                "указывает на слабую защиту"

            ),

            'negative': (

                "Низкая доля успешных атак в регионе "

                "указывает на хорошую защиту"

            ),

        },

        'success_rate_by_type': {

            'positive': (

                "Высокая доля успешных атак в отрасли "

                "указывает на системные уязвимости"

            ),

            'negative': (

                "Низкая доля успешных атак в отрасли "

                "указывает на зрелую ИБ-практику"

            ),

        },

    }

    direction = 'positive' if shap_value > 0 else 'negative'

    if feature_name in templates:

        return templates[feature_name][direction]

    if shap_value > 0:

        return f"Признак «{feature_name}» повышает риск"

    return f"Признак «{feature_name}» снижает риск"


6.7. Этап 5: Экспорт артефактов (Часы 38-42)
# ml/src/pipelines/export.py

import joblib

import json

from datetime import datetime

from pathlib import Path

def export_model(

    model,

    model_name: str,

    version: str,

    features: list,

    target: str,

    metrics: dict,

    encoders: dict = None,

    explainer=None,

    output_dir: str = 'models/v1',

):

    """

    Экспортирует модель и все сопутствующие артефакты.

    Backend загружает эти файлы при старте.

    """

    out = Path(output_dir)

    out.mkdir(parents=True, exist_ok=True)

    # 1. Модель

    model_path = out / f'{model_name}.pkl'

    joblib.dump(model, model_path)

    # 2. Энкодеры (если есть)

    if encoders:

        encoders_path = out / f'{model_name}_encoders.pkl'

        joblib.dump(encoders, encoders_path)

    # 3. SHAP explainer (если есть)

    if explainer:

        explainer_path = out / f'{model_name}_explainer.pkl'

        joblib.dump(explainer, explainer_path)

    # 4. Метаданные

    metadata = {

        'model_name': model_name,

        'version': version,

        'trained_at': datetime.utcnow().isoformat() + 'Z',

        'framework': type(model).__module__.split('.')[0],

        'algorithm': type(model).__name__,

        'target': target,

        'features': features,

        'feature_count': len(features),

    }

    with open(out / f'{model_name}_metadata.json', 'w') as f:

        json.dump(metadata, f, indent=2, ensure_ascii=False)

    # 5. Метрики

    with open(out / f'{model_name}_metrics.json', 'w') as f:

        json.dump(metrics, f, indent=2)

    # 6. Feature schema (для backend валидации)

    feature_schema = {

        'features': {

            name: 'categorical' if name in [

                'enterprise_type', 'region', 'season'

            ] else 'numeric'

            for name in features

        },

        'required': features,

    }

    with open(out / f'{model_name}_feature_schema.json', 'w') as f:

        json.dump(feature_schema, f, indent=2, ensure_ascii=False)

    print(f'Exported {model_name} to {out}/')

    return str(model_path)
Итоговая структура папки models/v1:
models/v1/

├── incident_classifier.pkl

├── incident_classifier_explainer.pkl

├── incident_classifier_metadata.json

├── incident_classifier_metrics.json

├── incident_classifier_feature_schema.json

│

├── attack_time_bucket.pkl

├── attack_time_bucket_metadata.json

├── attack_time_bucket_metrics.json

├── attack_time_bucket_feature_schema.json

│

├── attack_time_hour.pkl

├── attack_time_hour_metadata.json

├── attack_time_hour_metrics.json

├── attack_time_hour_feature_schema.json

│

├── threat_type.pkl

├── threat_type_encoders.pkl

├── threat_type_metadata.json

├── threat_type_metrics.json

├── threat_type_feature_schema.json

│

├── label_mappings.json          # Все маппинги меток

└── README.md                    # Описание моделей для команды


6.8. Сводная таблица всех 6 моделей
#
Модель
Алгоритм
Target
Тип задачи
Ключевые фичи
Целевая метрика
1
incident_classifier
CatBoost
success (0/1)
Бинарная классификация
enterprise_type, host_count, region, temporal, aggregated
ROC-AUC > 0.75
2a
attack_time_bucket
CatBoost
time_bucket (4 класса)
Мультикласс
enterprise_type, host_count, region, temporal
Accuracy > 0.40
2b
attack_time_hour
CatBoost Regressor
incident_hour (0-23)
Регрессия
enterprise_type, host_count, region, temporal
MAE < 5 часов
3
threat_type
LightGBM
threat_code (~100 классов)
Мультикласс
enterprise_type, host_count, region, temporal, aggregated
Accuracy@3 > 0.45
4
target_object
Rule-based маппинг
impact_object (10 категорий)
Детерминированный маппинг
threat_code → thrlist.impact_object
100% (маппинг)
5
vulnerability
Scoring formula
vuln_level (4 уровня)
Эвристическая формула
incident_prob, host_count, success_rates, cia_sum, recency
Экспертная валидация
6
recommendation
Rule engine + scoring
rec_code (top-3)
Ранжирование
threat_code, target_object, vuln_level, time_bucket
Релевантность top-3



6.9. Порядок работы ML-инженера по часам
Часы
Задача
Результат
0-2
Настройка окружения, загрузка данных
Jupyter notebook с raw данными
2-8
EDA + очистка + канонизация
data/processed/*.parquet, docs/DATA_CLEANING.md
8-14
Feature engineering
features/ модуль, feature_table.parquet
14-20
Модель 1 (incident) + Модель 2 (time)
2 обученные модели + метрики
20-26
Модель 3 (threat) + Модель 4 (object mapping)
1 модель + маппинг-словарь
26-32
Модель 5 (vulnerability scoring) + Модель 6 (recommendations)
Формула + rule engine
32-38
SHAP-объяснения для Модели 1
explainer.pkl + shap_utils.py
38-42
Экспорт всех артефактов
models/v1/ полностью заполнена
42-48
Интеграция с backend, тестирование, документация
Всё работает end-to-end


# DOCKER COMPOSE

# docker-compose.yml

version: '3.8'

services:

  db:

    image: postgres:15-alpine

    container_name: cyber-db

    environment:

      POSTGRES_DB: cyberdb

      POSTGRES_USER: cyberuser

      POSTGRES_PASSWORD: cyberpass

    ports:

      - "5432:5432"

    volumes:

      - ./db/init:/docker-entrypoint-initdb.d

      - pgdata:/var/lib/postgresql/data

    healthcheck:

      test: ["CMD-SHELL", "pg_isready -U cyberuser -d cyberdb"]

      interval: 5s

      timeout: 5s

      retries: 5

  backend:

    build:

      context: ./backend

      dockerfile: Dockerfile

    container_name: cyber-backend

    depends_on:

      db:

        condition: service_healthy

    environment:

      DATABASE_URL: postgresql://cyberuser:cyberpass@db:5432/cyberdb

      MODEL_DIR: /app/models/v1

      LOG_LEVEL: INFO

      CORS_ORIGINS: http://localhost:3000

    volumes:

      - ./models:/app/models:ro

      - ./data/processed:/app/data:ro

    ports:

      - "8000:8000"

    healthcheck:

      test: ["CMD", "curl", "-f", "http://localhost:8000/api/v1/health"]

      interval: 10s

      timeout: 5s

      retries: 3

  frontend:

    build:

      context: ./frontend

      dockerfile: Dockerfile

    container_name: cyber-frontend

    depends_on:

      backend:

        condition: service_healthy

    environment:

      VITE_API_URL: http://localhost:8000

    ports:

      - "3000:3000"

volumes:

  pgdata:


ЧАСТЬ VIII. ТАЙМЛАЙН 48 ЧАСОВ (СВОДНЫЙ)
ЧАС 0 ──────────────────────────────────────────────── ЧАС 8

│                                                        │

│  BACKEND: FastAPI скелет + моки всех 11 endpoints      │

│  FRONTEND: Vite + Router + Layout + пустые 5 страниц   │

│  ML: EDA + очистка данных + канонизация                │

│  DEVOPS: docker-compose + PostgreSQL + init.sql         │

│                                                        │

│  РЕЗУЛЬТАТ: Фронт делает запросы к мокам               │

│             БД заполнена чистыми данными                │

│                                                        │

ЧАС 8 ──────────────────────────────────────────────── ЧАС 24

│                                                        │

│  BACKEND: Реальные SQL-запросы для аналитики            │

│           SQLAlchemy ORM + repositories                 │

│  FRONTEND: Dashboard с графиками (на моках)             │

│            PredictionForm (на моках)                    │

│  ML: Feature engineering + Модели 1-3 обучены           │

│                                                        │

│  РЕЗУЛЬТАТ: Dashboard показывает реальную аналитику    │

│             3 модели обучены и экспортированы           │

│                                                        │

ЧАС 24 ─────────────────────────────────────────────── ЧАС 36

│                                                        │

│  BACKEND: Интеграция ML моделей (замена моков)          │

│           SHAP + рекомендации + сохранение в БД         │

│  FRONTEND: Prediction Page с реальными данными          │

│            Vulnerability + Recommendations pages        │

│  ML: Модели 4-6 + SHAP explainer + экспорт             │

│                                                        │

│  РЕЗУЛЬТАТ: POST /predict возвращает реальный прогноз  │

│             UI показывает SHAP-объяснения               │

│                                                        │

ЧАС 36 ─────────────────────────────────────────────── ЧАС 48

│                                                        │

│  ВСЕ: Полировка, демо-сценарии, обработка ошибок       │

│                                                        │

│  BACKEND: Error handling, логирование, демо-endpoints   │

│  FRONTEND: Loading/Error states, кнопка ДЕМО,           │

│            адаптивность, Toast-уведомления              │

│  ML: Документация метрик, README моделей                │

│  DEVOPS: docker-compose up на чистой машине,            │

│          README.md с инструкцией                        │

│                                                        │

│  РЕЗУЛЬТАТ: Готовый продукт для 5-минутного демо       │

└────────────────────────────────────────────────────────┘

# ФОРМАТ ОШИБОК (ЕДИНЫЙ ДЛЯ ВСЕХ ENDPOINTS)
// 400 Bad Request

{

    "error": {

        "code": "BAD_REQUEST",

        "message": "Некорректный формат запроса",

        "details": {}

    }

}

// 404 Not Found

{

    "error": {

        "code": "NOT_FOUND",

        "message": "Ресурс не найден",

        "details": {

            "resource": "threat",

            "id": "999"

        }

    }

}

// 422 Validation Error

{

    "error": {

        "code": "VALIDATION_ERROR",

        "message": "Ошибка валидации входных данных",

        "details": [

            {

                "field": "host_count",

                "message": "Значение должно быть больше 0",

                "received": -5

            }

        ]

    }

}

// 500 Internal Server Error

{

    "error": {

        "code": "INTERNAL_ERROR",

        "message": "Внутренняя ошибка сервера",

        "details": {}

    }

}

// 503 ML Service Unavailable

{

    "error": {

        "code": "ML_SERVICE_UNAVAILABLE",

        "message": "Аналитический модуль временно недоступен",

        "details": {

            "fallback_available": true

        }

    }

}



Конец документа. Версия 1.0.